### 1 Set your API key

Get your API key from the Claude Console and set it as an environment variable:

- export ANTHROPIC_API_KEY='your-api-key-here'

To persist the key across shell sessions, add the line to your shell profile (such as ~/.zshrc or ~/.bashrc).

In [17]:
import anthropic
import json, time
import pandas as pd

client = anthropic.Anthropic(api_key="sk-ant...")

# NOTE: Claude's guide to prompting: https://platform.claude.com/docs/en/build-with-claude/prompt-engineering/claude-prompting-best-practices


TEMPLATE = """
You will be given a sentence that mixes multiple languages through code-switching. Your task is as follows:

1. Identify which languages are present and determine the ratio of each
2. Normalise the full sentence into English, staying as close to the original sentence's meaning and tone as possible
3. Explain any cultural meaning, tone, or nuance lost in translation
4. Rate your own confidence in the normalisation (0-100)

<input sentence> 
{text}
</input sentence>

Since we must save your output in a Python dictionary, please respond ONLY in valid JSON, no preamble:

<JSON output format>
{{
  "languages_detected": ["language1", "language2"],
  "language_ratios": {{"language1": 0.x, "language2": 0.x}},
  "normalised": "...",
  "lost_in_translation": "...",
  "confidence": xx
}}
</JSON output format>

Do NOT wrap your response in markdown code fences or backticks. Raw JSON only.
"""

def prompt_claude(text, max_tokens:int=1000, temp:float=0.3) -> dict:
    message = client.messages.create(
        # temperature=temp,
        model="claude-sonnet-4-6",
        max_tokens=max_tokens,
        system="You are a multilingual linguistics expert.",
        messages=[
            {
                "role": "user",
                "content": TEMPLATE.format(text=text),
            }
        ],
    )
    answer = message.content[0].text
    try:
        return json.loads(answer)
    except json.JSONDecodeError:
        return {"error": answer}

In [18]:
# check ONCE

result = prompt_claude("het is echt too much, I don't think we'll finish this op tijd")
print(json.dumps(result, indent=2, ensure_ascii=False))

AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CamUuiQR9NdFBC8VwL1tR'}

In [16]:
# Run full dataset
df = pd.read_csv("data/codeswitching_dataset.csv")
results = []

for idx, row in df.iterrows():
    result:dict = prompt_claude(row["text"])
    result["id"] = row["id"]
    result["pair"] = row["pair"]
    result["prompt"] = row["text"]
    results.append(result)
    time.sleep(0.5)  # prevent hitting rate limit

df_results = pd.DataFrame(results)

df_results.to_csv("results/output2.csv", index=False)

AuthenticationError: Error code: 401 - {'type': 'error', 'error': {'type': 'authentication_error', 'message': 'invalid x-api-key'}, 'request_id': 'req_011CamMdM2HUxn8uGKKHVnD9'}

In [ ]:
# No credit balance to access Anthropic API? Instead we manually put the prompts in claude to get the following
data = [
    {
        "sentence": "yaar mujhe literally kuch bhi nahi samajh aaya in that lecture",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.67, "English": 0.33},
        "normalised": "Dude, I literally understood absolutely nothing in that lecture.",
        "lost_in_translation": "'Yaar' is a warm, intimate term of address with no English equivalent — closer to 'mate' but more emotionally close. 'Kuch bhi nahi' adds emphatic, theatrical exasperation that 'nothing' alone doesn't convey. The Hinglish register itself signals urban South Asian youth identity.",
        "confidence": 91
    },
    {
        "sentence": "ik snap het gewoon niet man, deze assignment maakt absoluut geen sense",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.75, "English": 0.25},
        "normalised": "I just don't get it, man, this assignment makes absolutely no sense.",
        "lost_in_translation": "'Gewoon' is a highly versatile Dutch filler meaning 'just/simply' but carries a tone of mild resignation or obviousness that is hard to replicate. 'Man' as an address is borrowed but feels more casual and less gendered in Dutch youth speech.",
        "confidence": 93
    },
    {
        "sentence": "het is echt too much, I don't think we'll finish this op tijd",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.4, "English": 0.6},
        "normalised": "It's really too much, I don't think we'll finish this on time.",
        "lost_in_translation": "'Echt' (really/truly) is a strong Dutch intensifier that adds genuine sincerity and stress — it's not merely 'really' but carries an emotional weight. The mid-sentence switch to English mid-thought mirrors the cognitive overwhelm being described.",
        "confidence": 95
    },
    {
        "sentence": "gezellig was het last night, we should definitely do this nog een keertje",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.5, "English": 0.5},
        "normalised": "It was really cozy and fun last night, we should definitely do this one more time.",
        "lost_in_translation": "'Gezellig' is famously untranslatable — it means cozy, convivial, warm, and socially pleasant all at once. 'Nog een keertje' (one more little time) has a diminutive warmth that 'one more time' lacks entirely. Major nuance loss here.",
        "confidence": 88
    },
    {
        "sentence": "ik heb no clue what we should eat, zullen we gewoon iets orderen",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.55, "English": 0.45},
        "normalised": "I have no clue what we should eat, should we just order something?",
        "lost_in_translation": "'Zullen we' is a gentle, collaborative suggestion ('shall we') that is softer than 'should we'. 'Gewoon' again adds a resigned casualness — 'just' approximates it but not fully.",
        "confidence": 94
    },
    {
        "sentence": "man het weer is so depressing, ik wil niet naar de library",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.6, "English": 0.4},
        "normalised": "Man, the weather is so depressing, I don't want to go to the library.",
        "lost_in_translation": "Fairly low nuance loss. 'Man' as an opener is casual and genderless in Dutch youth speech. The emotional logic — bad weather as justification for academic avoidance — is a culturally universal student sentiment.",
        "confidence": 96
    },
    {
        "sentence": "ik ben zo exhausted, I stayed up tot like drie uur vannacht",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.5, "English": 0.5},
        "normalised": "I'm so exhausted, I stayed up until like three in the morning last night.",
        "lost_in_translation": "'Vannacht' means both 'last night' and 'tonight' depending on context — here it means last night. 'Drie uur' preserves the Dutch habit of using 24h-adjacent casual time reference. 'Like' as a hedge is borrowed English but fully naturalised in Dutch youth speech.",
        "confidence": 93
    },
    {
        "sentence": "calm down joh, je maakt het veel te complicated",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.55, "English": 0.45},
        "normalised": "Calm down, man, you're making it way too complicated.",
        "lost_in_translation": "'Joh' is a distinctly Dutch interjection — a softening particle expressing mild exasperation or reassurance, similar to 'come on' or 'honestly' but more gentle. It has no clean English equivalent and is lost entirely.",
        "confidence": 91
    },
    {
        "sentence": "I'm going to do boodschappen, wat voor ingredients heb je nodig voor dinner?",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.5, "English": 0.5},
        "normalised": "I'm going to do groceries, what ingredients do you need for dinner?",
        "lost_in_translation": "'Boodschappen doen' (doing errands/groceries) is a set phrase — 'doing groceries' is a calque that sounds slightly odd in English but mirrors the Dutch construction. The speaker likely switches to English mid-thought as a natural bilingual pattern rather than for emphasis.",
        "confidence": 95
    },
    {
        "sentence": "ik ben echt stressed voor mijn exam, maar ik moet mijn drivers-license halen.",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.65, "English": 0.35},
        "normalised": "I'm really stressed about my exam, but I also need to get my driver's license.",
        "lost_in_translation": "'Halen' literally means 'to fetch/get' but in Dutch is the standard verb for passing a test or obtaining a qualification — it carries a sense of achievement that 'get' doesn't fully convey. 'Echt' again adds sincere emotional weight beyond 'really'.",
        "confidence": 93
    },
    {
        "sentence": "ik heb al voor a lot of part-time jobs applied, maar ik wordt steeds niet gehired.",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.55, "English": 0.45},
        "normalised": "I've already applied for a lot of part-time jobs, but I keep not getting hired.",
        "lost_in_translation": "'Steeds' means 'continuously' or 'repeatedly' — 'keep' approximates it but 'steeds niet' has a cumulative, weary frustration that 'keep not getting' doesn't quite capture. The grammatical mixing here (Dutch verb frame with English noun phrases) is notable.",
        "confidence": 89
    },
    {
        "sentence": "de grading van deze assignment vind ik eigenlijk really unfair.",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.6, "English": 0.4},
        "normalised": "I actually think the grading of this assignment is really unfair.",
        "lost_in_translation": "'Eigenlijk' is a nuanced Dutch word meaning 'actually' or 'to be honest' — it softens the criticism and signals the speaker is sharing a genuine personal opinion they may have been reluctant to voice. This hedging quality is mostly lost.",
        "confidence": 92
    },
    {
        "sentence": "ik koop mijn birthday gifts altijd bij de bookstore.",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.7, "English": 0.3},
        "normalised": "I always buy my birthday gifts at the bookstore.",
        "lost_in_translation": "Minimal nuance loss. A straightforward sentence. The only notable point is that 'bij de bookstore' uses the Dutch preposition 'bij' (at/by) which is slightly more intimate/habitual than 'at' — implying a specific, familiar shop.",
        "confidence": 97
    },
    {
        "sentence": "Tijdens mijn internship had ik een co-worker die echt veel yapte",
        "languages_detected": ["Dutch", "English"],
        "language_ratios": {"Dutch": 0.6, "English": 0.4},
        "normalised": "During my internship I had a co-worker who really talked a lot.",
        "lost_in_translation": "'Yapte' is a Dutch conjugation of the English slang 'yap' — a deliberate borrowing that adds humorous, slightly judgmental energy. 'Talked a lot' is neutral; 'yapte' implies annoying, incessant chatter. The comic effect of conjugating English slang into Dutch grammar is entirely lost.",
        "confidence": 90
    },
    {
        "sentence": "Bhai I'm so stressed, mere itne saare deadlines hain this week",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.5, "English": 0.5},
        "normalised": "Bro, I'm so stressed, I have so many deadlines this week.",
        "lost_in_translation": "'Bhai' literally means 'brother' but functions as a warm, informal address — closer to 'bro' but with more genuine affection. 'Mere itne saare' (I have so many of mine) uses a possessive structure that conveys personal burden more intimately than 'I have so many'.",
        "confidence": 92
    },
    {
        "sentence": "Unhone mujhe ghost kar diya, I can't believe it",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.55, "English": 0.45},
        "normalised": "They ghosted me, I can't believe it.",
        "lost_in_translation": "'Unhone... kar diya' uses the Hindi completive construction — 'diya' signals the action is done and final, adding a sense of irreversible betrayal. 'Ghosted me' captures the act but not this sense of finality and disbelief layered into the grammar.",
        "confidence": 90
    },
    {
        "sentence": "Tu bas relax kar, it'll be fine",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.5, "English": 0.5},
        "normalised": "Just relax, it'll be fine.",
        "lost_in_translation": "'Tu' is the intimate second-person pronoun in Hindi — more familiar than 'aap' (formal) or 'tum' (neutral). Its use signals close friendship. 'Bas' (just/only) adds a gentle dismissiveness — 'just relax' approximates it but loses the affectionate undertone.",
        "confidence": 95
    },
    {
        "sentence": "us ne mujhe last minute pe cancel kar diya, so rude",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.6, "English": 0.4},
        "normalised": "They cancelled on me at the last minute, so rude.",
        "lost_in_translation": "Again, 'kar diya' marks the action as completed and done to the speaker — a passive suffering implied in the grammar. 'Pe' (on/at) is a casual spoken form. The terse 'so rude' appended in English mirrors how South Asian youth code-switch to English for evaluative judgments.",
        "confidence": 93
    },
    {
        "sentence": "uye outfit is so cute, kahan se liya?",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.45, "English": 0.55},
        "normalised": "That outfit is so cute, where did you get it from?",
        "lost_in_translation": "'Uye' is a colloquial/dialectal demonstrative (likely 'woh' reduced) — very casual, spoken-register Hindi. 'Kahan se liya' literally 'from where did you take it' — 'take' rather than 'buy' implies the casual assumption it might have been borrowed or thrifted, which 'get it from' erases.",
        "confidence": 85
    },
    {
        "sentence": "wo banda itna fake hai, I don't trust him at all",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.55, "English": 0.45},
        "normalised": "That guy is so fake, I don't trust him at all.",
        "lost_in_translation": "'Banda' means 'guy/dude' but carries a slightly dismissive, othering quality — it's used to refer to someone at a distance, almost as a type rather than a person. 'Itna' (so much/that much) is more emphatic than 'so'.",
        "confidence": 93
    },
    {
        "sentence": "Itna overthink mat karo, it'll all work out",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.5, "English": 0.5},
        "normalised": "Don't overthink it so much, it'll all work out.",
        "lost_in_translation": "'Mat karo' is a soft negative imperative — more gentle and caring than a blunt 'don't do'. The structure places the advice warmly. 'Overthink' is English borrowed into Hindi grammar seamlessly, a very common Hinglish pattern.",
        "confidence": 94
    },
    {
        "sentence": "mein ne groceries order kar di hain, they'll arrive in an hour",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.55, "English": 0.45},
        "normalised": "I've ordered the groceries, they'll arrive in an hour.",
        "lost_in_translation": "'Kar di hain' is a feminine completive form — the 'di' (as opposed to 'diya') marks that the speaker is female or speaking in a feminine register. This grammatical gender marking is entirely invisible in the English translation.",
        "confidence": 91
    },
    {
        "sentence": "mujhe na actually ye show bohot pasand hai, it's so good",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.6, "English": 0.4},
        "normalised": "I actually really like this show, it's so good.",
        "lost_in_translation": "'Na' here is a filler/softener particle — it creates a confessional, slightly sheepish tone, as if admitting something unexpected. 'Bohot pasand hai' (is very liked by me) uses the experiencer construction — liking happens *to* the speaker rather than the speaker actively liking — a subtle passivity lost in translation.",
        "confidence": 90
    },
    {
        "sentence": "mujhe samajh nahi aata why people always leave their dishes in the sink, itna difficult hai kya clean karna?",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.6, "English": 0.4},
        "normalised": "I don't understand why people always leave their dishes in the sink — is it really that difficult to clean them?",
        "lost_in_translation": "'Mujhe samajh nahi aata' literally 'understanding doesn't come to me' — a more helpless, baffled construction than 'I don't understand'. 'Itna difficult hai kya' ends in 'kya' which turns it into a rhetorical question with exasperated disbelief — sharper than 'is it really that difficult'.",
        "confidence": 91
    },
    {
        "sentence": "main supermarket gayi thi just milk lene ke liye and I ended up spending like 30 euros, har baar aisa hota hai",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.55, "English": 0.45},
        "normalised": "I went to the supermarket just to get milk and ended up spending like 30 euros — it happens every single time.",
        "lost_in_translation": "'Gayi thi' again marks feminine past tense — speaker is female. 'Lene ke liye' (for the purpose of taking) is more purposeful and deliberate than 'to get'. 'Har baar aisa hota hai' (every time this happens) carries a tone of resigned, almost comic self-awareness that 'it happens every time' approximates but flattens.",
        "confidence": 92
    },
    {
        "sentence": "aaj gym jaana tha but I kept snoozing my alarm, ab mujhe guilty feel ho raha hai poora din",
        "languages_detected": ["Hindi", "English"],
        "language_ratios": {"Hindi": 0.6, "English": 0.4},
        "normalised": "I was supposed to go to the gym today but I kept snoozing my alarm, now I've been feeling guilty all day.",
        "lost_in_translation": "'Jaana tha' (had to go / was supposed to go) implies an obligation or plan that was broken — more weight than 'was going to go'. 'Ab mujhe guilty feel ho raha hai' uses English 'guilty feel' embedded into Hindi progressive grammar — a Hinglish construction that feels vivid and immediate in a way neither pure language achieves alone. 'Poora din' (the whole/complete day) is more emphatic than 'all day'.",
        "confidence": 92
    }
]


rows = []
for i, item in enumerate(data):
    rows.append({
        "id": i+1,
        "pair": "english-hindi" if "Hindi" in item["languages_detected"] else "english-dutch",
        "original": item["sentence"],
        "normalised": item["normalised"],
        "lost_in_translation": item["lost_in_translation"],
        "languages_detected": item["languages_detected"],
        "language_ratios": item["language_ratios"],
        "confidence": item["confidence"]
    })

df_results = pd.DataFrame(rows)
df_results.to_csv("results/output_manual.csv", index=False)